In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_G.xpt"
csv_file = DATA_DIR / "ALQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 ALQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ141Q",
    "ALQ141U",
    "ALQ151",
    "ALQ155"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ALQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 5615:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 5,615 rows x 10 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value before validating
# exactly where and how often it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached ALQ_G.xpt:
#
# ALQ120Q = 854
# ALQ141Q = 1,943
#
# TOTAL = 2,797
#
#
# CDC confirms:
#
# ALQ120Q:
#
#     "ENTER '0' FOR NEVER."
#
#     Published:
#
#       0-350  = 4,181
#       777    = 2
#       999    = 5
#       missing = 1,427
#
#
# ALQ141Q:
#
#     "ENTER '0' FOR NONE."
#
#     Published:
#
#       0-220  = 3,326
#       777    = 2
#       999    = 6
#       missing = 2,281
#

EXPECTED_TINY_COUNTS = {
    "ALQ120Q": 854,
    "ALQ141Q": 1943
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated ALQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 2797:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


print(
    "\nPASS: Exact 2,797-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

#
# Restore ONLY:
#
#     ALQ120Q
#     ALQ141Q
#
# No global replacement.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 2797:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric dataset should contain.
#
# ONLY validated tiny values change to zero.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Always run this BEFORE Int64 conversion.
#
# ALQ_G contains NO genuine fractional variables.
#
# ALQ120Q / ALQ141Q are quantities, but the public
# release stores them as whole-number counts.
#
# If fractional values unexpectedly appear,
# STOP rather than rounding them.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values "
        "detected. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables "
    "exist in ALQ_G."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71915
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# ALQ_G contains no genuine decimal fields.
#
# Therefore all 10 variables can safely become
# nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 ALQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 12. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. CDC CHECK - ALQ101
# ============================================================

#
# Had at least 12 drinks in any one year?
#
# CDC:
#
# 1 = Yes          3577
# 2 = No           1397
# 7 = Refused         0
# 9 = Don't know      3
# Missing           638
#

check_frequency(
    "ALQ101",
    {
        1: 3577,
        2: 1397,
        9: 3
    },
    638
)


# ============================================================
# 14. CDC CHECK - ALQ110
# ============================================================

#
# Had at least 12 drinks in lifetime?
#
# CDC:
#
# 1 = Yes           611
# 2 = No            783
# 7 = Refused         0
# 9 = Don't know      6
# Missing          4215
#

check_frequency(
    "ALQ110",
    {
        1: 611,
        2: 783,
        9: 6
    },
    4215
)


# ============================================================
# 15. CDC CHECK - ALQ120Q
# ============================================================

#
# Frequency of alcohol drinking over past 12 months.
#
# CDC:
#
# 0-350 = 4,181
# 777   = 2
# 999   = 5
# Missing = 1,427
#
# Attached XPT zero restoration:
#
# zero = 854
#

s = df["ALQ120Q"]


print("\n--- ALQ120Q ---")


print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-350:",
    int(
        s.between(
            0,
            350
        ).sum()
    )
)


print(
    "777:",
    int(
        s.eq(777)
        .sum()
    )
)


print(
    "999:",
    int(
        s.eq(999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 854
    and
    int(
        s.between(
            0,
            350
        ).sum()
    )
    == 4181
    and
    int(
        s.eq(777)
        .sum()
    )
    == 2
    and
    int(
        s.eq(999)
        .sum()
    )
    == 5
    and
    int(
        s.isna()
        .sum()
    )
    == 1427
):

    raise ValueError(
        "ALQ120Q differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: ALQ120Q matches CDC exactly."
)


# ============================================================
# 16. CDC CHECK - ALQ120U
# ============================================================

#
# Unit:
#
# 1 = Week
# 2 = Month
# 3 = Year
#

check_frequency(
    "ALQ120U",
    {
        1: 1371,
        2: 924,
        3: 1032
    },
    2288
)


# ============================================================
# 17. CDC CHECK - ALQ130
# ============================================================

#
# Average drinks/day over past 12 months.
#
# CDC:
#
# 1-82 = 3,329
# 777  = 2
# 999  = 3
# Missing = 2,281
#

s = df["ALQ130"]


if not (
    int(
        s.between(
            1,
            82
        ).sum()
    )
    == 3329
    and
    int(
        s.eq(777)
        .sum()
    )
    == 2
    and
    int(
        s.eq(999)
        .sum()
    )
    == 3
    and
    int(
        s.isna()
        .sum()
    )
    == 2281
):

    raise ValueError(
        "ALQ130 differs from CDC."
    )


print(
    "PASS: ALQ130 matches CDC."
)


# ============================================================
# 18. CDC CHECK - ALQ141Q
# ============================================================

#
# Number of days with 4/5+ drinks.
#
# CDC:
#
# 0-220 = 3,326
# 777   = 2
# 999   = 6
# Missing = 2,281
#
# Attached XPT:
#
# zero = 1,943
#

s = df["ALQ141Q"]


print("\n--- ALQ141Q ---")


print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-220:",
    int(
        s.between(
            0,
            220
        ).sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 1943
    and
    int(
        s.between(
            0,
            220
        ).sum()
    )
    == 3326
    and
    int(
        s.eq(777)
        .sum()
    )
    == 2
    and
    int(
        s.eq(999)
        .sum()
    )
    == 6
    and
    int(
        s.isna()
        .sum()
    )
    == 2281
):

    raise ValueError(
        "ALQ141Q differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: ALQ141Q matches CDC exactly."
)


# ============================================================
# 19. CDC CHECK - ALQ141U
# ============================================================

check_frequency(
    "ALQ141U",
    {
        1: 395,
        2: 315,
        3: 673
    },
    4232
)


# ============================================================
# 20. CDC CHECK - ALQ151
# ============================================================

check_frequency(
    "ALQ151",
    {
        1: 655,
        2: 3528,
        7: 1,
        9: 2
    },
    1429
)


# ============================================================
# 21. CDC CHECK - ALQ155
# ============================================================

#
# Years drinking 4/5+ drinks almost every day.
#
# CDC:
#
# 1-60 = 588
# 666 = Less than 1 year = 64
# 777 = Refused = 0
# 999 = Don't know = 3
# Missing = 4,960
#
# Important:
#
# Interview instruction says:
#
#     IF LESS THAN 1 YEAR, ENTER 0
#
# but CDC stores this in the released dataset as:
#
#     666
#
# Therefore ALQ155 should NOT contain zero.
#

s = df["ALQ155"]


if not (
    int(
        s.between(
            1,
            60
        ).sum()
    )
    == 588
    and
    int(
        s.eq(666)
        .sum()
    )
    == 64
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 3
    and
    int(
        s.eq(0)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 4960
):

    raise ValueError(
        "ALQ155 differs from CDC."
    )


print(
    "PASS: ALQ155 matches CDC."
)


# ============================================================
# 22. ALLOWED RELEASED CODE CHECK
# ============================================================

#
# Additional safeguard against unexpected codes.
#

CATEGORICAL_ALLOWED_VALUES = {

    "ALQ101": {
        1,
        2,
        9
    },

    "ALQ110": {
        1,
        2,
        9
    },

    "ALQ120U": {
        1,
        2,
        3
    },

    "ALQ141U": {
        1,
        2,
        3
    },

    "ALQ151": {
        1,
        2,
        7,
        9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed_values in CATEGORICAL_ALLOWED_VALUES.items():

    observed_values = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected_values = (
        observed_values
        -
        allowed_values
    )


    if unexpected_values:

        raise ValueError(
            f"{col} contains unexpected values: "
            f"{sorted(unexpected_values)}"
        )


    print(
        f"PASS: {col} contains only expected "
        "released codes."
    )


# ============================================================
# 23. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY expected changes from raw pandas XPT:
#
# ALQ120Q:
#     854 TINY_VALUE -> 0
#
# ALQ141Q:
#     1,943 TINY_VALUE -> 0
#
# NOTHING else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 24. EXPORT CSV
# ============================================================

#
# No rounding.
# No float_format.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 25. RE-READ CSV
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 26. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV columns round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 27. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 28. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 29. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 30. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# ALQ_G contains NO genuine decimal variables.
#
# Therefore ".0" should not appear in any populated field
# in this particular CSV.
#
# IMPORTANT:
#
# Do NOT copy this global .0 rule to Dietary files or
# Lab/Exam files containing genuine decimal measurements.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 31. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 32. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71915
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 33. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "ALQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "ALQ120Q zero restorations:",
    854
)


print(
    "ALQ141Q zero restorations:",
    1943
)


print(
    "Total zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 ALQ_G XPT -> CSV
Rows: 5,615
Columns: 10
PASS: Structure = 5,615 rows x 10 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 2,797

Tiny values by variable:
ALQ141Q    1943
ALQ120Q     854

PASS: Exact 2,797-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
ALQ120Q: 854 restored to 0
ALQ141Q: 1,943 restored to 0

PASS: 2,797 validated zero values restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
ALQ101: 0 fractional observations
ALQ110: 0 fractional observations
ALQ120Q: 0 fractional observations
ALQ120U: 0 fractional observations
ALQ130: 0 fractional observations
ALQ141Q: 0 fractional observations
ALQ141U: 0 fractional observations
ALQ151: 0 fractional observations
ALQ155: 0 fractional observations
PASS: No genuine fractional variables exist in ALQ_G.

--- SEQN ---
Minimum: 62161
Maximum: 71915
Missing: 0
Duplicate